In [1]:
import sys
from pathlib import Path
# Get project root (current working directory when running from project root)
# If running from research/, use parent; otherwise use cwd()
project_root = Path.cwd() if (Path.cwd() / 'vault').exists() else Path.cwd().parent
sys.path.insert(0, str(project_root))

In [2]:
from utils.enums import Ticker, TimeFrame, Direction
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from feature_extraction.feature_extractor import extract_features_for_bias_node
from feature_selection.base_models import BaseModel, QuantileBinningModel, TwoBinBinningModel
from ensemble.vault_manager import (
    initialize_vault, create_ensemble_directory, 
    load_feature_base_models, list_features
)
from eda.feature_explorer import FeatureExplorer
import utils.helpers as helpers

/home/raman/repos/Trading-Algo/utils/permutation_test/permute_bars.py:38: UserWarning: feature_selection.opt_thresh not found. Using simplified threshold optimization. For full functionality, ensure the opt_thresh module is available.
  warnings.warn(


# RSI Bias Node Test - Lookback = 2

This notebook demonstrates the simplified workflow using the new helper functions:
1. **One-line feature extraction**: Using `extract_features_for_bias_node()` to extract features with forward returns automatically
2. **Self-contained exploration**: Using `FeatureExplorer` which handles binning model fitting internally
3. **Comprehensive analysis**: Using `generate_summary_report()` to run all relevant analysis methods at once
4. **Vault operations**: Creating and saving BaseModel to vault for production use

## Key Improvements:
- **Simplified workflow**: No manual forward returns calculation or BaseModel fitting for exploration
- **Automatic alignment**: Features and targets are automatically aligned
- **Self-contained**: FeatureExplorer handles model fitting internally
- **Comprehensive**: Single method call generates full analysis report

In [3]:
# Step 1: Extract features with forward returns automatically
# The new helper function handles everything: feature extraction + forward returns computation
# Define tickers for this ensemble (used throughout notebook)
#ensemble_tickers = [Ticker.NQ, Ticker.ES, Ticker.RTY, Ticker.YM]
from typing import List

ensemble_tickers = [Ticker.ES, Ticker.NQ, Ticker.YM, Ticker.RTY] 
# ensemble_tickers = [
#     Ticker.ES,  # E-Mini S&P
#     Ticker.NQ,  # E-Mini Nasdaq
#     Ticker.YM,  # E-Mini Dow
#     Ticker.RTY, # E-Mini Russell 2000
#     Ticker.CL,  # Crude Oil (WTI)
#     Ticker.HO,  # ULSD (Heating Oil)
#     Ticker.GC,  # Gold
#     Ticker.HG,  # Copper
#     Ticker.SI,  # Silver
#     Ticker.PL,  # Platinum
#     Ticker.EU,  # EURUSD
#     Ticker.JY,  # Japanese Yen
#     Ticker.BP,  # British Pound
#     Ticker.CD,  # Canadian Dollar
#     Ticker.SF,  # Swiss Franc
#     Ticker.C,   # Corn
#     Ticker.S,   # Soybeans
#     Ticker.W,   # Wheat (SRW)
#     Ticker.GF,  # Feeder Cattle
#     Ticker.TY,  # 10Yr T.Note
#     Ticker.FV,  # 5Yr T.Note
#     Ticker.US,  # 30Yr T.Bond
#     Ticker.TU   # 2Yr T.Note
# ]

# All tickers from utils/enums.py (Equity Indices, Energy, Metals, Currencies, Agricultural, Meat, Fixed Income)

#Define bias node specification
bias_spec = {
    'module_name': 'ultimate_c',
    'timeframes': [TimeFrame.D],
    'params': {'lookback': [2], 'smooth_lookback': [3], 'factor': [3]}
}

# bias_spec = {
#     'module_name': 'prev_return',
#     'timeframes': [TimeFrame.D],
#     'params': {}
# }

# Extract features with forward returns in one call!
# This automatically:
# 1. Loads candles for all tickers
# 2. Computes forward returns (avoiding lookahead bias)
# 3. Extracts features
# 4. Aligns features and targets correctly
print("Extracting features with forward returns...")
features_df, targets_df = extract_features_for_bias_node(
    bias_spec=bias_spec,
    ticker=ensemble_tickers,
    start=datetime(2000, 1, 1),
    end=datetime(2024, 12, 31),
    use_millisecond_offset=True, 
    target_col='log_return'
)

print(f"\n✓ Features extracted successfully!")
print(f"  Features shape: {features_df.shape}")
print(f"  Targets shape: {targets_df.shape}")
print(f"  Feature columns: {[col for col in features_df.columns if col != 'ticker']}")
print(f"  Target columns: {[col for col in targets_df.columns if col != 'ticker']}")
print(f"  Unique tickers: {features_df['ticker'].unique() if 'ticker' in features_df.columns else 'N/A'}")
print(f"\n✓ Forward returns calculated correctly:")
print(f"  - Return at timestamp t = return from t to t+1")
print(f"  - Feature at timestamp t predicts return[t] (forward return)")
print(f"  - No lookahead bias: feature uses data up to t, predicts return from t to t+1")

Extracting features with forward returns...
⚠ Cython node optimizations not available (compile with: python utils/setup_cython.py build_ext --inplace)

✓ Features extracted successfully!
  Features shape: (20294, 6)
  Targets shape: (20294, 5)
  Feature columns: ['ultimate_c_signal_D_factor_3_lookback_2_smoothLookback_3', 'atr_atr_D_period_252', 'atr_atr_pct_D_period_252', 'ewsd_daily_pct', 'ewsd_annual_pct']
  Target columns: ['raw_return', 'log_return', 'log_return_atr', 'log_return_ewsd']
  Unique tickers: ['ES' 'NQ' 'YM' 'RTY']

✓ Forward returns calculated correctly:
  - Return at timestamp t = return from t to t+1
  - Feature at timestamp t predicts return[t] (forward return)
  - No lookahead bias: feature uses data up to t, predicts return from t to t+1


In [4]:
# Step 2: Create FeatureExplorer for analysis
# Build metadata for FeatureExplorer (optional but recommended)
metadata = {'feature_metadata': {}}
for feature_col in features_df.columns:
    if feature_col != 'ticker':
        parsed = helpers.parse_feature_column_name(feature_col)
        metadata['feature_metadata'][feature_col] = {
            'module': parsed.get('module'),
            'parameters': parsed.get('params', {}),
            'timeframes': [parsed.get('tf')] if parsed.get('tf') else [],
            'base_name': parsed.get('module'),
            'full_name': feature_col
        }

# Create FeatureExplorer (self-contained - no BaseModel needed!)
explorer = FeatureExplorer(
    features_df=features_df,
    targets_df=targets_df,
    metadata=metadata
)

print("FeatureExplorer created successfully!")
print(f"  Number of features: {explorer.n_features}")
print(f"  Feature names: {explorer.feature_names}")
print(f"  Number of samples: {explorer.n_samples}")
print(f"  Date range: {explorer.date_range[0].date()} to {explorer.date_range[1].date()}")


FeatureExplorer created successfully!
  Number of features: 1
  Feature names: ['ultimate_c_signal_D_factor_3_lookback_2_smoothLookback_3']
  Number of samples: 20294
  Date range: 2000-01-03 to 2024-12-30


In [5]:


binning_model = QuantileBinningModel(n_bins=10, selection_metric='sortino', strategy='long')
#binning_model = TwoBinBinningModel( selection_metric='sortino', strategy='short')

print("\nGenerating comprehensive summary report with export...")
results = explorer.generate_summary_report(
    binning_model=binning_model,
    target_col='log_return',
    strategy='long',
    show_plots=False,  # Set to False when exporting
    verbose=True,
    export_report=True,  # Export report to files
    export_path='./reports/summary_report',  # Export directory
    permutation_test_nreps=100,
    include_ticker_plots=False
)



Generating comprehensive summary report with export...

Generating Comprehensive Feature Analysis Report

[1/10] Computing basic summary statistics...
  ✓ Computed statistics for 1 features

[2/10] Computing feature-target correlations...
  ✓ Computed correlations for 1 features

[3/10] Computing intra-feature correlations...
  ⚠ Skipping intra-feature correlations: Need at least 2 numeric features for correlation matrix

[4/10] Plotting decile analysis...
  ✓ Generated 1 decile plots

[5/10] Plotting signal-gated cumulative returns...
  ✓ Generated 1 signal plots

[6/10] Analyzing parameter sensitivity...
[DEBUG] analyze_parameter: 'ultimate_c_signal_D_factor_3_lookback_2_smoothLookback_3' samples=20294 (feature NaNs=0, target NaNs=0)

Debug Info:
Feature group: ('ultimate', 'd')
Features found: ['ultimate_c_signal_D_factor_3_lookback_2_smoothLookback_3']
Available columns in features_df: ['ultimate_c_signal_D_factor_3_lookback_2_smoothLookback_3', 'ticker', 'atr_atr_D_period_252', '

In [ ]:

# Step 7: Create BaseModel for vault operations (if needed)
# BaseModel is only needed when saving to vault
print("\n" + "="*70)
print("Creating BaseModel for vault operations...")
print("="*70)

binning_model_vault = QuantileBinningModel(n_bins=10, selection_metric='sortino', strategy='long')
#binning_model_vault = TwoBinBinningModel( selection_metric='sortino', strategy='short')

base_model = BaseModel(
    feature_config={'bias_node_spec': bias_spec},
    binning_model=binning_model_vault,
    tickers=ensemble_tickers
)

print("BaseModel created for vault operations!")
print(f"  Ticker: {base_model.ticker.name}")
print(f"  Bias node spec: {base_model.bias_node_spec}")
print(f"  Number of bias nodes: {len(base_model.bias_nodes)}")

# Note: BaseModel doesn't need to be fitted before saving to vault
# It can be fitted later before deployment

print("\nCreating ensemble directory...")
ensemble_dir = create_ensemble_directory(
    timeframe=TimeFrame.D,
    ensemble_name='ultimate_c_2_3_3 ',
    direction=Direction.LONG,
    tickers=ensemble_tickers
)

print(f"✓ Created ensemble directory: {ensemble_dir}")
print(f"  Ensemble tickers: {[t.name for t in ensemble_tickers]}")
print(f"  Default ensemble directory is now set automatically")

In [ ]:
# Step 8: Save base model to vault
print("\nSaving base model to vault...")
# Pass all ensemble tickers when saving
# ensemble_dir is optional - uses default set by create_ensemble_directory()
model_id = base_model.save_to_vault(tickers=ensemble_tickers)
print(f"✓ Saved model with ID: {model_id}")
print(f"  Feature column: {base_model.feature_column}")
print(f"  Model ID: {model_id}")
print(f"  Saved with tickers: {[t.name for t in ensemble_tickers]}")

In [ ]:
# Step 9: Verify vault save/load
print("\nLoading models from vault...")
# ensemble_dir is optional - uses default set by create_ensemble_directory()
# Path resolution automatically handles running from research/ directory
loaded_models = load_feature_base_models(feature_column=base_model.feature_column)
model_id = 'quantile_binning_2'

print(f"✓ Loaded {len(loaded_models)} model(s)")
print(f"  Model keys (ticker, model_id): {list(loaded_models.keys())}")

# Verify loaded model (keys are now (ticker, model_id) tuples)
key = (Ticker.ES, model_id)
if key in loaded_models:
    loaded_model = loaded_models[key]
    print(f"\n✓ Model '{model_id}' for {Ticker.ES.name} loaded successfully")
    print(f"  Binning model fitted: {loaded_model.binning_model.is_fitted_}")
    print(f"  Best long bin: {loaded_model.binning_model.best_long_bin_}")
    print(f"  Ticker: {loaded_model.ticker.name}")
else:
    print(f"\n✗ Model '{model_id}' not found. Available keys: {list(loaded_models.keys())}")

# List all features in ensemble
print("\n" + "="*70)
print("Features in ensemble:")
print("="*70)
# ensemble_dir is optional - uses default set by create_ensemble_directory()
features_list = list_features()
print(features_list)